<a href="https://colab.research.google.com/github/kvnxmars/cfg-vs-llm-mcq-synthesis/blob/main/LLM_Regex_Synthesiser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating the Computational Efficiency, Formal Correctness, and Pedagogical Realism of LLM-Synthesized Regular Expression Assessment Materials

This notebook implements the complete research pipeline for evaluating and comparing **Symbolic Context-Free Grammar (CFG) MCQ generation** against **Probabilistic Neural Generation (using the GPT-5.6 Luna model)**.

### Pipeline Architecture:
1. **Cell 1 & 2: Environment Setup and Dataset Ingestion:** Installs dependencies and parses the ground-truth benchmark questions from Ade-Ibijola's (2018) 10,000 MCQ dataset.
2. **Cell 3: Pedagogical MCQ Wrappers:** Configures the prompt layouts for **Figure 7 (Classical Single-Choice MCQ)** and **Figure 8 (Multi-Answer MCQ)**.
3. **Cell 4: Neural Synthesis & Custom Cost Logging:** Orchestrates parallel LLM generation runs using GPT-5.6 Luna, implementing token-level cost tracking (including caching) and latency metrics.
Cell 1 (Code)

In [11]:
#environment setup & libraries
print("Installing libraries..")

#install opneai for gpt4o prompting and fpdf2 for pdf generation
!pip install --quiet openai fpdf2 pypdf pdfplumber
print("Libraries successfully installed!")

print("Assembling imports..")
import os
import re
import sys
import time
import random
import pypdf
import json

from google.colab import files

print("Setup complete.. Ready to load pdf.")

Installing libraries..
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.3 MB/s eta 0:00:00
Libraries successfully installed!
Assembling imports..
Setup complete.. Ready to load pdf.


## Cell 2: Ground-Truth Dataset Parsing and Text Normalization


This cell ingests both uploaded baseline PDFs and merges them into a single, unified database (`baseline_tasks.json`) of exactly **500 benchmark tasks**.

The current pipeline extracts:

A. **Single-Answer Baseline:** Parses the first **250 tasks** from `10000 iterations of MCQs and Solutions.pdf` (Figure 7 layout).

B. **Multi-Answer Baseline:** Parses the first **250 tasks** from `1000 iterations of MCQs - multiple correct solutions.pdf` (Figure 8 layout).

### Normalization and Cleansing:
* **Asterisk Normalization:** Converts mathematical Unicode stars (\\(\ast\\) or `\u2217`) to standard keyboard stars (`*`) to ensure compatibility with our automata engine.
* **Alphabet Boundary Restricting:** Dynamically extracts the restricted alphabet (\\(\Sigma\\)) for each question, filtering out spaces and empty-string lambda symbols (\\(\lambda\\)).
* **Deferred Multi-Answer Keys:** For multi-answer tasks, `labeled_correct` is initialized to `null`. Cell 5 will programmatically grade and label these options using our local DFA verifier.






In [12]:
# ============================================================
# CELL 2A: PARSE 10,000-ITEM SINGLE-ANSWER PDF (Figure 7)
# ============================================================
import os, re, json
import pdfplumber
from google.colab import drive, files

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
OUTPUT_DIR   = "/content/drive/MyDrive/regex_llm_project"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SINGLE_PDF = "10000 iterations of MCQs and Solutions.pdf"
if not os.path.exists(SINGLE_PDF):
  print(f"Please upload '{SINGLE_PDF}':")
  files.upload()

def clean_regex_text(raw):
  return re.sub(r"\s+", "", raw).replace("*", "*")

print("Extracting text (column-aware, ~5s for 36 pages)..")
text_parts = []
with pdfplumber.open(SINGLE_PDF) as pdf:
    for page in pdf.pages:
      w, h = page.width, page.height
      text_parts.append(page.crop((0, 0, w / 2, h)).extract_text() or "")
      text_parts.append(page.crop((w / 2, 0, w, h)).extract_text() or "")
text = "\n".join(text_parts)

# A wildcard can never consume the start of the next "(N)." item -- a malformed
# item fails cleanly instead of silently swallowing its neighbor's fields.
NB = r"(?:(?!\(\d+\)\.).)*?"
pattern = re.compile(
    r"\((\d+)\)\.\s*Rex\s*=\s*(" + NB + r");\s*Σ\s*=\s*\{([^}\n]+)\}\.\s*"
    r"(" + NB + r")Answer:\s*([A-E])\.",
    re.DOTALL
)
opt_pattern = re.compile(r"([A-E])\.\s*(.*?)\s*(?=(?:[A-E]\.)|$)", re.DOTALL)

all_matches = list(pattern.finditer(text))
matched_ids = {int(m.group(1)) for m in all_matches}
skipped = sorted(set(range(1, 10001)) - matched_ids)

COUNT = 250
tasks = []
for m in all_matches:
    task_id = int(m.group(1))
    target_regex = clean_regex_text(m.group(2))
    alphabet = [s.strip() for s in m.group(3).replace(" ", "").split(",") if s.strip() and s.strip() != "λ"]
    answer_letter = m.group(5)
    option_block = m.group(4)
    letter_to_text = {letter: opt_text.strip() for letter, opt_text in opt_pattern.findall(option_block)}
    options = [{"string": letter_to_text.get(l, ""), "labeled_correct": (l == answer_letter)} for l in "ABCDE"]

    tasks.append({
        "task_id": task_id,
        "wrapper_type": "Figure 7 (Single-Choice)",
        "target_regex": target_regex,
        "alphabet": alphabet,
        "options": options
    })
    if len(tasks) >= COUNT:
        break

out_path = os.path.join(OUTPUT_DIR, "single_tasks.json")
with open(out_path, "w") as f:
    json.dump(tasks, f, indent=2)

# ---- SNIPPET ONLY ----
print(f"\n✅ Matched {len(all_matches)}/10000 items ({len(skipped)} malformed & excluded: {skipped[:10]}{'...' if len(skipped) > 10 else ''})")
print(f"💾 Saved first {len(tasks)} tasks to '{out_path}'\n")
print("--- Spot check (compare directly against the PDF page) ---")
for t in tasks[:3]:
    correct = next(o["string"] for o in t["options"] if o["labeled_correct"])
    print(f"  #{t['task_id']}: r = {t['target_regex']}  Σ={t['alphabet']}  →  correct: \"{correct}\"")


Mounted at /content/drive
Exctracting text (column-aware, ~5s for 36 pages)..

✅ Matched 10000/10000 items (0 malformed & excluded: [])
💾 Saved first 250 tasks to '/content/drive/MyDrive/regex_llm_project/single_tasks.json'

--- Spot check (compare directly against the PDF page) ---
  #1: r = f+j∗  Σ=['l', 'f', 'j']  →  correct: "ffff"
  #2: r = b∗f?  Σ=['z', 'r', 'b', 'f']  →  correct: "bbb"
  #3: r = q+i?  Σ=['y', 'n', 'q', 'i']  →  correct: "qi"


In [13]:
# ============================================================
# CELL 2B: PARSE 1,000-ITEM MULTI-ANSWER PDF (Figure 8) + MERGE
# ============================================================
import os, re, json
import pdfplumber
from google.colab import files

MULTI_PDF = "1000 iterations of MCQs - multiple correct solutions.pdf"
if not os.path.exists(MULTI_PDF):
    print(f"Please upload '{MULTI_PDF}':")
    files.upload()

def clean_regex_text(raw):
    return re.sub(r"\s+", "", raw).replace("∗", "*")

print("🔄 Extracting text (column-aware, ~5s for 36 pages)...")
text_parts = []
with pdfplumber.open(MULTI_PDF) as pdf:
    for page in pdf.pages:
        w, h = page.width, page.height
        text_parts.append(page.crop((0, 0, w / 2, h)).extract_text() or "")
        text_parts.append(page.crop((w / 2, 0, w, h)).extract_text() or "")
text = "\n".join(text_parts)

NB = r"(?:(?!\(No\.\d+\)\.).)*?"
pattern = re.compile(
    r"\(No\.\s*(\d+)\)\.\s*Let\s*r\s*=\s*(" + NB + r")\.\s*"
    r"Which\s*of\s*the\s*following\s*strings\s*is\s*in\s*L\s*\(r\)\?\s*Take\s*"
    r"Σ\s*=\s*\{([^}\n]+)\}\.\s*"
    r"(" + NB + r")Circle\s*your\s*answers\.",
    re.DOTALL
)

all_matches = list(pattern.finditer(text))
matched_ids = {int(m.group(1)) for m in all_matches}
skipped = sorted(set(range(1, 1001)) - matched_ids)

# These answers are UNDERLINED in the PDF, not text-labeled. pdfplumber's text
# extraction can't see underline formatting, so labeled_correct stays None here
# and is resolved programmatically by the DFA engine in Cell 5.
COUNT = 250
tasks = []
for m in all_matches:
    task_id = int(m.group(1)) + 10000   # offset, avoids colliding with single-answer IDs
    target_regex = clean_regex_text(m.group(2))
    alphabet = [s.strip() for s in m.group(3).replace(" ", "").split(",") if s.strip() and s.strip() != "λ"]
    opts = [o.strip() for o in m.group(4).strip().rstrip(",").split(",") if o.strip()]
    options = [{"string": o, "labeled_correct": None} for o in opts]

    tasks.append({
        "task_id": task_id,
        "wrapper_type": "Figure 8 (Multi-Answer)",
        "target_regex": target_regex,
        "alphabet": alphabet,
        "options": options
    })
    if len(tasks) >= COUNT:
        break

OUTPUT_DIR = "/content/drive/MyDrive/regex_llm_project"
out_path = os.path.join(OUTPUT_DIR, "multi_tasks.json")
with open(out_path, "w") as f:
    json.dump(tasks, f, indent=2)

print(f"\n✅ Matched {len(all_matches)}/1000 items ({len(skipped)} malformed & excluded: {skipped})")
print(f"💾 Saved first {len(tasks)} tasks to '{out_path}'\n")
print("--- Spot check (labeled_correct is null by design) ---")
for t in tasks[:3]:
    print(f"  #{t['task_id']}: r = {t['target_regex']}  Σ={t['alphabet']}  options={[o['string'] for o in t['options']]}")

# ---- MERGE with Cell 2A's output into the unified baseline ----
with open(os.path.join(OUTPUT_DIR, "single_tasks.json")) as f:
    single_tasks = json.load(f)

unified = single_tasks + tasks
baseline_path = os.path.join(OUTPUT_DIR, "baseline_tasks.json")
with open(baseline_path, "w") as f:
    json.dump(unified, f, indent=2)

print(f"\n💾 Merged {len(single_tasks)} single + {len(tasks)} multi = {len(unified)} → '{baseline_path}'")

🔄 Extracting text (column-aware, ~5s for 36 pages)...

✅ Matched 992/1000 items (8 malformed & excluded: [4, 49, 82, 96, 171, 185, 678, 692])
💾 Saved first 250 tasks to '/content/drive/MyDrive/regex_llm_project/multi_tasks.json'

--- Spot check (labeled_correct is null by design) ---
  #10001: r = g*yi+  Σ=['e', 't', 'w', 'g', 'y', 'i']  options=['ettttii', 'twwwgggyyiiii', 'eetttwwggi', 'ttwwwyyi', 'gyiiii']
  #10002: r = l*f?  Σ=['h', 'y', 'l', 'f']  options=['yy', 'h', 'llllf', 'l', 'yyyyf']
  #10003: r = l+vd*  Σ=['n', 'y', 'p', 'l', 'v', 'd']  options=['nnppp', 'llvd', 'yyllvvd', 'llllv', 'nnyyypppllllvvdd']

💾 Merged 250 single + 250 multi = 500 → '/content/drive/MyDrive/regex_llm_project/baseline_tasks.json'


## Cell 3: Pedagogical MCQ Wrappers &amp; Strict Operator Constraints

This cell configures the **System Prompt** and **User Prompt Templates** for Figure 7 and Figure 8 MCQ wrappers.
### FLA Operator Constraints:
To isolate the neural model's logical reasoning engine rather than its memory of programming-specific regex shorthand (such as \`\\d\`, \`\\w\`, or anchors like \`^\` and \`$\`), the system prompt strictly constrains the LLM to use only core theoretical operators:

\* Concatenation (implied)

\* Alternation / Union (\`|\`)

\* Kleene Star (\`\*\`)

\* Kleene Plus (\`+\`)

\* Optionality (\`?\`)

\* Explicit grouping parentheses (\`()\`)

In [14]:
# =====================================================================
# CELL 3: MCQ WRAPPERS & SYSTEM PROMPTS (FIGURE 7 & FIGURE 8)
# =====================================================================

SYSTEM_PROMPT = """
You are an expert tutor in Formal Languages and Automata (FLA) theory. Your task is to generate a pedagogically sound Multiple-Choice Question (MCQ) item for an undergraduate Computer Science practice exam.

You must strictly adhere to the following language and syntax constraints:
1. ALPHABET CONSTRAINT (Σ): You may ONLY use the characters explicitly provided in the user's alphabet. No other characters or shorthand symbols (such as \\d, \\w, \\s) are allowed.
2. OPERATOR CONSTRAINT: You may ONLY use the core theoretical operators of regular expressions:
   - Concatenation (implied, e.g., ab)
   - Alternation / Union (using "|" or "+", e.g., a|b)
   - Kleene Star ("*")
   - Kleene Plus ("+")
   - Optionality ("?")
   - Explicit grouping parentheses ("( )")
   Do NOT use lookarounds, backreferences, anchors (^ or $), or any PCRE programming extensions.

You must respond with STRICT JSON ONLY. Do not wrap the JSON in markdown code fences, and do not include any conversational preamble or postscript.

JSON SCHEMA:
{
  "synthesized_regex": "<the regular expression r_LLM you constructed>",
  "shuffled_options": [
    {"label": "A", "string": "<string_1>", "labeled_correct": true},
    {"label": "B", "string": "<string_2>", "labeled_correct": false},
    {"label": "C", "string": "<string_3>", "labeled_correct": false},
    {"label": "D", "string": "<string_4>", "labeled_correct": false},
    {"label": "E", "string": "<string_5>", "labeled_correct": false}
  ],
  "pedagogical_rationales": {
    "regex_design": "Explain in 1 sentence how you constructed the regex.",
    "option_mapping": {
       "A": "Why this string matches/does not match.",
       "B": "Why this string matches/does not match.",
       "C": "Why this string matches/does not match.",
       "D": "Why this string matches/does not match.",
       "E": "Why this string matches/does not match."
    }
  }
}
"""

# Wrapper 1: Figure 7 Classic Format
WRAPPER_FIG7_TEMPLATE = """
Target Language Specification: {description}
Alphabet (Σ): {{{alphabet_symbols}}}

Generate one classical MCQ item following this exact problem wrapper layout from Figure 7:

"For each of the following regular expressions, defined over the indicated alphabet, determine (from options A to E) which string is accepted by the expression.
Let r = {target_regex}. Take Σ = {{{alphabet_symbols}}}.

[List five randomly shuffled strings here; four wrong and one correct answer.]"

Instructions:
1. Synthesize the regular expression (it must be equivalent to {target_regex}).
2. Generate exactly FIVE options (labeled A to E).
3. Exactly ONE option must be mathematically correct (labeled_correct = true), and the other FOUR must be incorrect distractors.

Provide your output strictly matching the required JSON schema.
"""

# Wrapper 2: Figure 8 Multi-Answer Format
WRAPPER_FIG8_TEMPLATE = """
Target Language Specification: {description}
Alphabet (Σ): {{{alphabet_symbols}}}
Number of Correct Options (m): {num_correct}

Generate one multi-answer MCQ item following this exact problem wrapper layout from Figure 8:

"Let r = [synthesized_regex]. Which of the following strings is in L(r)? Take Σ = {{{alphabet_symbols}}}.
[List 5 randomly shuffled strings here; {num_correct} correct strings, and {num_wrong} wrong strings.]"

Instructions:
1. Synthesize the regular expression (it must be equivalent to {target_regex}).
2. Generate exactly FIVE options (labeled A to E).
3. Exactly {num_correct} options must be mathematically correct (labeled_correct = true), and the remaining {num_wrong} options must be incorrect distractors (where {num_correct} must be between 1 and 4).

Provide your output strictly matching the required JSON schema.
"""

print("✅ Step 3 complete! Prompt templates for Figure 7 and Figure 8 are compiled.")

✅ Step 3 complete! Prompt templates for Figure 7 and Figure 8 are compiled.




## Cell 4A: Free Llama-3.1 Synthesis Pipeline (Groq Engine)

This cell executes the 500-task benchmark using **Meta's Llama-3.1-70B** hosted on Groq's high-speed LPU infrastructure.

### Features:
- **Zero-Cost Execution:** Runs entirely on Groq's free developer tier (API key from console.groq.com).
- **High Throughput:** Executes at ~250 tokens/second without artificial daily rate limit caps.
- **Dedicated Output:** Saves results to `cumulative_tasks_llama.json` to preserve data for comparative model analysis against OpenAI.






In [15]:
# =====================================================================
# CELL 4A: FREE GROQ (LLAMA / GPT-OSS) TEST-SET / SYNTHESIS LOOP
# =====================================================================
import os, sys, time, json, random
from openai import OpenAI
from getpass import getpass

# -- 0. Mount Drive so incremental progress survives a disconnect --
# Colab's local disk (/content/...) is fine WITHIN a session — that's
# how your earlier runs saved cumulative_tasks_llama.json successfully.
# But if the runtime disconnects (idle timeout, ~90min job), that disk
# is wiped. Mount Drive once and write there instead.
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/regex_llm_project"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -- 1. Groq Provider Configuration --
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = "openai/gpt-oss-120b"   # <-- STRING, not a list

GROQ_API_KEY = getpass("Enter Groq API key (console.groq.com): ")
groq_client = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)

# -- 2. TEST-SET LIMIT --
# None = full 500-task run. Set to 5 or 100 to eyeball output before committing.
MAX_TASKS = 5

# -- 3. Ingest baseline dataset --
baseline_path = os.path.join(OUTPUT_DIR, "baseline_tasks.json")
if not os.path.exists(baseline_path) and os.path.exists("baseline_tasks.json"):
    baseline_path = "baseline_tasks.json"   # fallback if Cell 2 hasn't been pointed at Drive yet

try:
    with open(baseline_path, "r") as f:
        all_baseline_tasks = json.load(f)
    print(f"Loaded {len(all_baseline_tasks)} benchmark tasks from '{baseline_path}'.")
except FileNotFoundError:
    print("Error! baseline_tasks.json missing. Run Cell 2 first.")
    sys.exit(1)

# -- 4. Resume log: ONLY successful tasks are ever written here --
llama_output_file = os.path.join(OUTPUT_DIR, "cumulative_tasks_llama.json")
llama_dataset = []
if os.path.exists(llama_output_file):
    with open(llama_output_file, "r") as f:
        llama_dataset = json.load(f)
    print(f"Resuming: {len(llama_dataset)} tasks already completed.")
existing_ids = {item["task_id"] for item in llama_dataset}

# -- 5. Execution loop --
print(f"\nLaunching {GROQ_MODEL} (Groq) Synthesis Loop..")
print("----------------------------------------------")

processed_this_run = 0
for idx, task in enumerate(all_baseline_tasks):
    if MAX_TASKS is not None and processed_this_run >= MAX_TASKS:
        print(f"\nHit MAX_TASKS={MAX_TASKS}. Stopping for review.")
        break

    task_id = task["task_id"]
    if task_id in existing_ids:
        continue

    target_regex = task["target_regex"]
    alphabet_list = task["alphabet"]
    alphabet_str = ", ".join(alphabet_list)
    description = f"strings matching the regular expression {target_regex}"
    wrapper_type = task["wrapper_type"]

    if wrapper_type == "Figure 7 (Single-Choice)":
        user_prompt = WRAPPER_FIG7_TEMPLATE.format(
            description=description, target_regex=target_regex, alphabet_symbols=alphabet_str
        )
    else:
        rng = random.Random(task_id)          # seeded -> identical prompt every run/model
        num_correct = rng.randint(2, 4)
        user_prompt = WRAPPER_FIG8_TEMPLATE.format(
            description=description, target_regex=target_regex, alphabet_symbols=alphabet_str,
            num_correct=num_correct, num_wrong=5 - num_correct
        )

    print(f"[{GROQ_MODEL}] Task {idx + 1}/{len(all_baseline_tasks)} [ID: {task_id}] | {wrapper_type}...")

    parsed_response = None
    prompt_tokens = completion_tokens = 0
    parse_error = None

    for attempt in range(4):
        t0 = time.perf_counter()
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            duration = time.perf_counter() - t0
            prompt_tokens = response.usage.prompt_tokens
            completion_tokens = response.usage.completion_tokens
            raw_content = response.choices[0].message.content.strip()   # <-- [0] added
            parsed_response = json.loads(raw_content)
            parse_error = None
            break
        except Exception as e:
            duration = time.perf_counter() - t0
            parse_error = str(e)
            if attempt < 3:
                wait = 2 * (2 ** attempt)
                print(f"  attempt {attempt+1} failed ({parse_error[:80]}); retry in {wait}s")
                time.sleep(wait)

    if parsed_response is None:
        print(f"  GIVING UP on task {task_id}: {parse_error}")
        continue   # NOT added to existing_ids -> will retry next run

    record = {
        "task_id": task_id,
        "model_engine": GROQ_MODEL,
        "wrapper_type": wrapper_type,
        "target_regex": target_regex,
        "alphabet": alphabet_list,
        "synthesized_regex": parsed_response.get("synthesized_regex"),
        "options": parsed_response.get("shuffled_options"),
        "rationales": parsed_response.get("pedagogical_rationales"),
        "metrics": {
            "duration_s": duration,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "parse_error": None,
        }
    }

    llama_dataset.append(record)
    existing_ids.add(task_id)
    processed_this_run += 1

    with open(llama_output_file, "w") as f:
        json.dump(llama_dataset, f, indent=4)

    print(f"  {duration:.2f}s | in: {prompt_tokens} | out: {completion_tokens}")
    time.sleep(0.5)
    print("-------------------------------------------------------------")

print(f"\nRun complete! {len(llama_dataset)} tasks saved to '{llama_output_file}'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter Groq API key (console.groq.com): ··········
Loaded 500 benchmark tasks from '/content/drive/MyDrive/regex_llm_project/baseline_tasks.json'.

Launching openai/gpt-oss-120b (Groq) Synthesis Loop..
----------------------------------------------
[openai/gpt-oss-120b] Task 1/500 [ID: 1] | Figure 7 (Single-Choice)...
  2.20s | in: 763 | out: 941
-------------------------------------------------------------
[openai/gpt-oss-120b] Task 2/500 [ID: 2] | Figure 7 (Single-Choice)...
  2.54s | in: 768 | out: 1131
-------------------------------------------------------------
[openai/gpt-oss-120b] Task 3/500 [ID: 3] | Figure 7 (Single-Choice)...
  2.23s | in: 762 | out: 969
-------------------------------------------------------------
[openai/gpt-oss-120b] Task 4/500 [ID: 4] | Figure 7 (Single-Choice)...
  2.84s | in: 766 | out: 1068
-----------------------------------

This cell is to test the results of the test set experiment (take a peek into the dataset).

In [16]:
import json
with open(os.path.join(OUTPUT_DIR, "cumulative_tasks_llama.json")) as f:
    print(json.dumps(json.load(f)[:2], indent=2))

[
  {
    "task_id": 1,
    "model_engine": "openai/gpt-oss-120b",
    "wrapper_type": "Figure 7 (Single-Choice)",
    "target_regex": "f+j\u2217",
    "alphabet": [
      "l",
      "f",
      "j"
    ],
    "synthesized_regex": "(f|j*)",
    "options": [
      {
        "label": "A",
        "string": "l",
        "labeled_correct": false
      },
      {
        "label": "B",
        "string": "jjj",
        "labeled_correct": true
      },
      {
        "label": "C",
        "string": "fl",
        "labeled_correct": false
      },
      {
        "label": "D",
        "string": "ff",
        "labeled_correct": false
      },
      {
        "label": "E",
        "string": "jf",
        "labeled_correct": false
      }
    ],
    "rationales": {
      "regex_design": "Combined the single\u2011character alternative f with j* to capture either f or any number of j's (including none).",
      "option_mapping": {
        "A": "Contains the symbol l, which is not allowed by the regex,